In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.integrate as integrate
from scipy import optimize
import plotly.graph_objects as go
import plotly.io as pio

In [ ]:
eta_0 = 120 * np.pi  # Free space impedance
class layer:
    def __init__(self, f, eps, mu, theta, polarization,l=0, l_in_parts_of_wavelength=False, array_mode=False):
        self.eps = eps
        self.mu = mu
        self.l = l
        self.polarization = polarization
        self.theta = theta
        self.f = f
        self.array_mode = array_mode
        # if array_mode:
        #     self.f = f[:, None, None]
        #     self.theta = theta[None, None, :]
        self.w = 2 * np.pi * f # [:,None, None]
        mu0 = 4e-7 * np.pi
        eps0 = 8.854e-12
        eta_0 = np.sqrt(mu0/eps0)
        self.n = np.sqrt(mu * eps)
        if polarization == 'vert':
            self.z = eta_0/self.eps * np.sqrt(self.n**2 - (np.sin(theta)**2)) #[None, None, :]
        elif polarization == 'horiz':
            self.z = eta_0 * self.mu/np.sqrt(self.n**2 - (np.sin(theta)**2)) #[None, None, :]
        self.beta = self.w/0.3 * np.sqrt(self.n**2 - (np.sin(theta)**2)) #[:, None, :]
        if l_in_parts_of_wavelength:
            self.l = l * 2*np.pi/self.beta
        self.phi = self.beta * self.l #[:, None, :]
    def abcd(self):
        abcd_res = np.array([[np.cos(self.phi), -1j*self.z*np.sin(self.phi)],
                        [-1j*(1/self.z)*np.sin(self.phi), np.cos(self.phi)]], dtype=complex)
        if self.array_mode:
            return abcd_res.transpose(2,3,4,0,1)
        return abcd_res

In [ ]:
eta_0 = 120 * np.pi  # Free space impedance
class impedance_sheet:
    def __init__(self, f, L, theta, array_mode=False):
        self.f = f
        self.theta = theta
        self.array_mode = array_mode
        if array_mode:
            self.like_theta = np.ones_like(self.theta)
            self.like_f = np.ones_like(self.f)
        self.w = 2 * np.pi * f
        mu0 = 4e-7 * np.pi
        eps0 = 8.854e-12
        eta_0 = np.sqrt(mu0/eps0)
        self.L = L
        self.z = 1j * self.w * L *1e9
    def abcd(self):
        abcd_res = np.array([[1*self.like_theta*self.like_f, 0*self.like_theta*self.like_f],
                        [-1/self.z*self.like_theta*self.like_f, 1*self.like_theta*self.like_f]], dtype=complex)
        if self.array_mode:
            return abcd_res.transpose(2,3,4,0,1)
        return abcd_res

In [ ]:
l = np.array([1, 2, 3])[:, None, None]
p = np.array([4, 5, 6])[None,:, None]
l2 = np.ones_like(l)
print(np.shape(l))  # (3, 1, 1)
print(np.shape(l2))  # (3, 1, 1)
print(np.shape(l * p))  # (3, 3, 1)
print(l + p) 

In [ ]:
def v_in(theta, phi, polarization):
    if polarization == 'vert':
        return np.cos(theta )*np.cos(phi)
    elif polarization == 'horiz':
        return np.sin(phi)
def z0(theta, polarization):
    if polarization == 'vert':
        return eta_0*np.cos(theta)
    elif polarization == 'horiz':
        return eta_0/np.cos(theta)

In [ ]:
for N in [2, 4, 6]:
    eps = np.ones(N)
    mu = np.ones(N)
    eps[1::2] = 10
    mu[::2] = 4
    theta = np.linspace(-np.pi/2, np.pi/2, 200)
    #theta = np.linspace(-1, 1, 1)
    #theta = np.array([0])
    #print(theta)
    f = 5
    l = np.zeros(N)
    for i in range(N):
        layer_i = layer(f, eps[i], mu[i], 0, 'vert', l=0.25, l_in_parts_of_wavelength=True)
        l[i] = layer_i.l
    l[0] *= 2
    E_res_all = []
    for theta_i in theta:
        E_res = 0
        for polarization in ['vert', 'horiz']:
            abcd_all = []
            for i in range(N):
                layer_i = layer(f, eps[i], mu[i], theta_i, polarization, l=l[i])
                abcd_all.append(layer_i.abcd())
                # print(layer_i.l/(0.3/f/layer_i.n))
                # print(layer_i.z)
                # print(layer_i.phi)
                # print(layer_i.abcd())
                
            v_in_i = v_in(theta_i, 0, polarization)
            z0_i = z0(theta_i, polarization)
            abcd_total = np.eye(2, dtype=complex)
            for abcd in abcd_all:
                abcd_total = np.matmul(abcd_total, abcd)
            #print(abcd_total)
            A, B, C, D = abcd_total.flatten()
            I_0 = (-C*B/A+D)*2*v_in_i/(z0_i-B/A)
            shift_layer = layer(f, eps[0], mu[0], theta_i, polarization, l=l[0]/2)
            abcd_shift = np.linalg.inv(shift_layer.abcd())
            A_s, B_s, C_s, D_s = abcd_shift.flatten()
            E_res += B_s*I_0
        E_res_all.append(np.abs(E_res)**2)
    E_res_all = np.array(E_res_all)
    E_res_all /= np.max(E_res_all)
    plt.ylim(-60, 0)
    plt.plot(theta*180/np.pi, 10*np.log10(E_res_all), label=f'N={N}')
    print('for N=', N, 'суммарная длина', sum(l)*1000, 'mm')
plt.legend()
plt.show()        
#print(E_res_all)
#'vert' отвечает за E-плоскость

In [ ]:
for N in [2, 4, 6]:
    eps = np.ones(N)
    mu = np.ones(N)
    eps[1::2] = 10
    mu[::2] = 4
    theta = np.linspace(-np.pi/2, np.pi/2, 200)
    #theta = np.linspace(-1, 1, 1)
    #theta = np.array([0])
    #print(theta)
    f = 5
    l = np.zeros(N)
    for i in range(N):
        layer_i = layer(f, eps[i], mu[i], 0, 'vert', l=0.25, l_in_parts_of_wavelength=True)
        l[i] = layer_i.l
    l[0] *= 2
    E_res_all = []
    for theta_i in theta:
        E_res = 0
        for polarization in ['horiz', 'vert']:
            abcd_all = []
            for i in range(N):
                layer_i = layer(f, eps[i], mu[i], theta_i, polarization, l=l[i])
                abcd_all.append(layer_i.abcd())
                # print(layer_i.l/(0.3/f/layer_i.n))
                # print(layer_i.z)
                # print(layer_i.phi)
                # print(layer_i.abcd())
                
            v_in_i = v_in(theta_i, 3*np.pi/4, polarization)
            z0_i = z0(theta_i, polarization)
            abcd_total = np.eye(2, dtype=complex)
            for abcd in abcd_all:
                abcd_total = np.matmul(abcd_total, abcd)
            #print(abcd_total)
            A, B, C, D = abcd_total.flatten()
            I_0 = (-C*B/A+D)*2*v_in_i/(z0_i-B/A)
            shift_layer = layer(f, eps[0], mu[0], theta_i, polarization, l=l[0]/2)
            abcd_shift = np.linalg.inv(shift_layer.abcd())
            A_s, B_s, C_s, D_s = abcd_shift.flatten()
            E_res += np.abs(B_s*I_0)**2
        E_res_all.append(np.abs(E_res)**2)
    E_res_all = np.array(E_res_all)
    E_res_all /= np.max(E_res_all)
    plt.ylim(-60, 0)
    plt.plot(theta*180/np.pi, 10*np.log10(E_res_all), label=f'N={N}')
    print('for N=', N, 'суммарная длина', sum(l)*1000, 'mm')
plt.show()        
#print(E_res_all)

In [ ]:
def e_on_direction(N, eps, mu, l, f=5, phi=0, theta=0): 
    E_res = 0
    for polarization in ['vert', 'horiz']:
        abcd_all = []
        for i in range(N):
            layer_i = layer(f, eps[i], mu[i], theta, polarization, l=l[i])
            abcd_all.append(layer_i.abcd())   
        v_in_i = v_in(theta, phi, polarization)
        z0_i = z0(theta, polarization)
        abcd_total = np.eye(2, dtype=complex)
        for abcd in abcd_all:
            abcd_total = np.matmul(abcd_total, abcd)
        A, B, C, D = abcd_total.flatten()
        I_0 = (-C*B/A+D)*2*v_in_i/(z0_i-B/A)
        shift_layer = layer(f, eps[0], mu[0], theta, polarization, l=l[0]/2)
        abcd_shift = np.linalg.inv(shift_layer.abcd())
        A_s, B_s, C_s, D_s = abcd_shift.flatten()
        E_res += np.abs(B_s*I_0)**2
    return np.sqrt(E_res)
def l_for_article_resonance_structure(N, eps, mu, f=5, theta = 0):
    l = np.zeros(N)
    for i in range(N):
        layer_i = layer(f, eps[i], mu[i], theta, 'vert', l=0.25, l_in_parts_of_wavelength=True)
        l[i] = layer_i.l
    l[0] *= 2
    return l

In [ ]:
# Теперь хорошо бы построить эти графики в стилистике моего диплома
import sys
sys.path.append(r'C:\Users\Michael\Desktop\Antenna_new\T_model_python')

from My_plotter import Plotter, Style

In [ ]:
st = Style()

fig, ax = plt.subplots()
pl = Plotter(ax, st)

for N in [2, 4, 6]:
    eps = np.ones(N)
    mu = np.ones(N)
    eps[1::2] = 10
    mu[::2] = 4
    theta = np.linspace(-np.pi/2*0.9999, np.pi/2*0.9999, 200)
    f = 5
    l = np.zeros(N)
    for i in range(N):
        layer_i = layer(f, eps[i], mu[i], 0, 'vert', l=0.25, l_in_parts_of_wavelength=True)
        l[i] = layer_i.l
    l[0] *= 2
    e_norm = np.abs(e_on_direction(N, eps, mu, l, phi=0, theta=0))
    print('e_norm =', e_norm)
    #radiation_pattern_e_plane = lambda theta: np.abs(e_on_direction(N, eps, mu, l, theta = theta, phi=0)/max_e)**2
    radiation_pattern = lambda phi, theta: np.abs(e_on_direction(N, eps, mu, l, phi=phi, theta=theta)/e_norm)**2 * np.sin(theta)
    res, err = integrate.dblquad(
        radiation_pattern,
        0, np.pi/2*0.9999,                # границы для x
        lambda x: 0,          # нижняя граница y
        lambda x: np.pi*2*0.9999,       # верхняя граница y
        epsrel=1e-2
    )
    #res_e, err_e = integrate.quad(radiation_pattern_e_plane, -np.pi/2, np.pi/2)
    #print('Directivity по E-плоскости =', 10*np.log10(2*np.pi/res_e), 'dBi')
    print(res, err)
    print('Directivity =', 10*np.log10(4*np.pi/res), 'dBi')
    theta = np.linspace(-np.pi/2*0.9999, np.pi/2*0.9999, 200)
    P = np.zeros(200)
    for i in range(len(theta)):
        P[i] = np.abs(e_on_direction(N, eps, mu, l, phi=0, theta=theta[i]))**2
    pl.plot(theta*180/np.pi, 10*np.log10(P/max(P)), label=f'N={N}, D={10*np.log10(4*np.pi/res):.2f} dBi')
pl.set_ylim((-60, 0))
pl.set_xlabel('$\\theta^\\circ$')
pl.set_ylabel('$Диаграмма направленности, dB$')
pl.finalize()
ax.legend(loc='upper right')
plt.show()

написать функцию, которая 

In [ ]:
def v_in(theta, phi, polarization):
    if polarization == 'vert':
        return np.cos(theta )*np.cos(phi)
    elif polarization == 'horiz':
        return np.sin(phi)
    
def z0(theta, polarization):
    if polarization == 'vert':
        return eta_0*np.cos(theta)
    elif polarization == 'horiz':
        return eta_0/np.cos(theta)

def e_on_direction_array(N, eps, mu, l, f, phi, theta): 
    E_res = np.zeros((f.size, phi.size, theta.size), dtype=complex)
    f = f[:, None, None] 
    phi = phi[None, :, None]
    theta = theta[None, None, :]
    for polarization in ['vert', 'horiz']:
        abcd_all = []
        for i in range(N):
            layer_i = layer(f, eps[i], mu[i], theta, polarization, l=l[i], array_mode=True)
            abcd_all.append(layer_i.abcd())   #layer_i.abcd() должен быть 5-мерным массивом
        v_in_i = v_in(theta, phi, polarization)
        z0_i = z0(theta, polarization)
        abcd_total = np.eye(2, dtype=complex)
        for abcd in abcd_all:
            abcd_total = np.matmul(abcd_total, abcd)
        abcd_total = np.transpose(abcd_total, (3,4,0,1,2))
        A, B, C, D = abcd_total[0][0], abcd_total[0][1], abcd_total[1][0], abcd_total[1][1]
        I_0 = (-C*B/A+D)*2*v_in_i/(z0_i-B/A)
        shift_layer = layer(f, eps[0], mu[0], theta, polarization, l=l[0]/2, array_mode=True)
        abcd_shift = np.linalg.inv(shift_layer.abcd())
        abcd_shift = np.transpose(abcd_shift, (3,4,0,1,2))
        B_s = abcd_shift[0][1]
        E_res += np.abs(B_s*I_0)**2
    return np.sqrt(E_res)

def e_on_direction_array_with_sheets(N, eps, mu, l, mask, L, f, phi, theta): 
    E_res = np.zeros((f.size, phi.size, theta.size), dtype=complex)
    f = f[:, None, None] 
    phi = phi[None, :, None]
    theta = theta[None, None, :]
    for polarization in ['vert', 'horiz']:
        abcd_all = []
        for i in range(2*N+1):
            if mask[i]:
                layer_i = impedance_sheet(f, L[i], theta, array_mode=True)
            else:
                layer_i = layer(f, eps[i], mu[i], theta, polarization, l=l[i], array_mode=True)
            abcd_all.append(layer_i.abcd())   #layer_i.abcd() должен быть 5-мерным массивом
        v_in_i = v_in(theta, phi, polarization)
        z0_i = z0(theta, polarization)
        abcd_total = np.eye(2, dtype=complex)
        for abcd in abcd_all:
            abcd_total = np.matmul(abcd_total, abcd)
        abcd_total = np.transpose(abcd_total, (3,4,0,1,2))
        A, B, C, D = abcd_total[0][0], abcd_total[0][1], abcd_total[1][0], abcd_total[1][1]
        I_0 = (-C*B/A+D)*2*v_in_i/(z0_i-B/A)
        shift_layer = layer(f, eps[0], mu[0], theta, polarization, l=0.01500, array_mode=True)
        abcd_shift = np.linalg.inv(shift_layer.abcd())
        abcd_shift = np.transpose(abcd_shift, (3,4,0,1,2))
        B_s = abcd_shift[0][1]
        E_res += np.abs(B_s*I_0)**2
    return np.sqrt(E_res)

In [ ]:
e1 = e_on_direction(4, np.array([1,10,1,10]), np.array([4,1,4,1]), l_for_article_resonance_structure(4, np.array([1,10,1,10]), np.array([4,1,4,1]), f=5, theta = 0), phi=0, theta=0, f=5)
print(e1)

In [ ]:
N = 2
eps = np.ones(2*N + 1)
mu = np.ones(2*N +1)
L = np.array([7.99446555e-9]*(2*N+1))
l = np.array([0.00614498]*(2*N+1))
l[0] = 0.02557249
l[2*N] = 0
a = ([False, True]*N)
a.append(False)
mask = np.array(a)
print(e_on_direction_array_with_sheets(N, eps, mu, l, mask, L, np.array([5, 5.1]), np.array([0, 0.1]), np.array([0, 0.1])))

In [ ]:
e2 = e_on_direction_array(6, np.array([1,10,1,10,1,10]), np.array([1,4,1,4,1,4]), l_for_article_resonance_structure(6, np.array([1,10,1,10,1,10]), np.array([1,4,1,4,1,4]), f=5, theta = 0), np.array([5, 10]), np.array([0]), np.array([0]))
print(e2)

In [ ]:
import pandas as pd
data_e = pd.read_csv('theor_impedance_sheet/E-plane.csv', header=None)
data_h = pd.read_csv('theor_impedance_sheet/H-plane.csv', header=None)

In [ ]:
num_points = 200
theta_arr = np.linspace(-np.pi/2*0.99, np.pi/2*0.99, num_points)
N = 1
eps = np.ones(2*N + 1)
mu = np.ones(2*N +1)
L = np.array([7.99446555e-9]*(2*N+1))
l = np.array([0.00614498]*(2*N+1))
l[0] = 0.02557249
l[2*N] = 0
a = ([False, True]*N)
a.append(False)
mask = np.array(a)
e = e_on_direction_array_with_sheets(N, eps, mu, l, mask, L, np.array([5.00]), np.array([0]), theta_arr)
h = e_on_direction_array_with_sheets(N, eps, mu, l, mask, L, np.array([5.00]), np.array([np.pi/2]), theta_arr)
plt.plot(theta_arr*180/np.pi, e[0][0].real/2, label = 'T-matrix_E-plane')
plt.plot(theta_arr*180/np.pi, h[0][0].real/2, label = 'T-matrix_H-plane')
plt.title('E-plane')
plt.plot(data_e[0], data_e[1]/2, 'ro', label = 'CST')
plt.plot(data_h[0], data_h[1]/2, 'go', label = 'CST_H-plane')
plt.xlabel('Angle (degrees)')
plt.ylabel('Electric Field (В/м)')
plt.legend()
plt.show()

In [ ]:
# Теперь хорошо бы построить эти графики в стилистике моего диплома
import sys
sys.path.append(r'C:\Users\Michael\Desktop\Antenna_new\T_model_python')

from My_plotter import Plotter, Style

In [ ]:
data_e = pd.read_csv('theor_impedance_sheet/E-plane.csv', header=None)
data_h = pd.read_csv('theor_impedance_sheet/H-plane.csv', header=None)

num_points = 200
theta_arr = np.linspace(-np.pi/2*0.99, np.pi/2*0.99, num_points)
N = 1
eps = np.ones(2*N + 1)
mu = np.ones(2*N +1)
L = np.array([7.99446555e-9]*(2*N+1))
l = np.array([0.00614498]*(2*N+1))
l[0] = 0.02557249
l[2*N] = 0
a = ([False, True]*N)
a.append(False)
mask = np.array(a)

e = e_on_direction_array_with_sheets(N, eps, mu, l, mask, L, np.array([5.00]), np.array([0]), theta_arr)
h = e_on_direction_array_with_sheets(N, eps, mu, l, mask, L, np.array([5.00]), np.array([np.pi/2]), theta_arr)

st = Style()
fig, ax = plt.subplots()
pl = Plotter(ax, st)

pl.plot(theta_arr*180/np.pi, e[0][0].real/2, label = 'Данный метод')

ax.plot((data_e[0])[::2], (data_e[1]/2)[::2], 'ro', label = 'CST')

pl.set_xlabel('$\\theta^\\circ$')
pl.set_ylabel('$\\kappa_{TM}$')
pl.set_ylim((0, 2.5))
pl.finalize()
plt.show()

fig, ax = plt.subplots()
pl = Plotter(ax, st)

pl.plot(theta_arr*180/np.pi, h[0][0].real/2, label = 'Данный метод')
ax.plot((data_h[0])[::2], (data_h[1]/2)[::2], 'ro', label = 'CST')
pl.set_xlabel('$\\theta^\\circ$')
pl.set_ylabel('$\\kappa_{TE}$')
pl.set_ylim((0, 2.5))
pl.finalize()
plt.show()

In [ ]:
import pandas as pd
data_e = pd.read_csv('theor_impedance_sheet/H-plane.csv', header=None)

In [ ]:
num_points = 200
theta_arr = np.linspace(-np.pi/2*0.99, np.pi/2*0.99, num_points)
N = 1
eps = np.ones(2*N + 1)
mu = np.ones(2*N +1)
L = np.array([7.99446555e-9]*(2*N+1))
l = np.array([0.00614498]*(2*N+1))
l[0] = 0.02557249
l[2*N] = 0
a = ([False, True]*N)
a.append(False)
mask = np.array(a)
e = e_on_direction_array_with_sheets(N, eps, mu, l, mask, L, np.array([5]), np.array([np.pi/2]), theta_arr)
plt.plot(theta_arr*180/np.pi, e[0][0].real, label = 'T-matrix')
plt.title('H-plane')
plt.plot(data_e[0], data_e[1], 'ro', label = 'CST')
plt.xlabel('Angle (degrees)')
plt.ylabel('Electric Field (В/м)')
plt.legend()
plt.show()

In [ ]:
N = 4
eps = np.ones(N)
mu = np.ones(N)
eps[1::2] = 10
mu[::2] = 4
theta = np.linspace(-np.pi/2.1, np.pi/2.1, 200)
phi = np.array([0, np.pi/2])
e3 = np.abs(e_on_direction_array(4, eps, mu, l_for_article_resonance_structure(N, eps, mu, f=5, theta = 0), np.array([5]), phi, theta))[0][0]
e4 = np.abs(e_on_direction_array(4, eps, mu, l_for_article_resonance_structure(N, eps, mu, f=5, theta = 0), np.array([5]), phi, theta))[0][1]
plt.plot(theta, 20*np.log10(e3/np.max(e3)))
plt.plot(theta, 20*np.log10(e4/np.max(e4)))
plt.ylim(-60, 0)
plt.show()

In [ ]:
e2 = e_on_direction_array(6, np.array([1,10,1,10,1,10]), np.array([1,4,1,4,1,4]), np.array([0.25,0.25,0.25,0.25,0.25,0.25])*0.3/5, np.array([5, 10]), np.array([0, 0.5]), np.array([0, 0.5]))

In [ ]:
a = np.array([1, 2, 3])[None, :]
b = np.array([4, 5, 6])[:, None]
c = np.array([1, 1, 1])[:, None]
d = np.array([0, 1, 0])[None, :]
print(a*b)
print(c*d)
print((a*b)@(c*d))
print(a*(a==2))

In [ ]:
a = np.array([3, -1, 1e-6, 2], dtype=float)
a[a<=1e-6] = 0
a[a>0] = np.log10(a[a>0])
print(a)

In [ ]:
def log_scale_dB(e, thresh_dB=-60):
    e = np.abs(e)
    threshhold = 10**(thresh_dB/20)*np.max(e)
    mask = e>threshhold
    mask_2 = e<=threshhold
    e[mask_2] = 0
    e[mask] = 20*np.log10(e[mask]/threshhold)
    return e

In [ ]:
def e_on_direction_with_sheets(N, eps, mu, l, mask, L, f, phi, theta):
    return e_on_direction_array_with_sheets(N, eps, mu, l, mask, L, np.array([f]), np.array([phi]), np.array([theta]))[0][0][0]

In [ ]:
N = 1
eps = np.ones(2*N + 1)
mu = np.ones(2*N +1)
L = np.array([7.99446555e-9]*(2*N+1))
l = np.array([0.00614498]*(2*N+1))
l[0] = 0.02557249
l[2*N] = 0
a = ([False, True]*N)
a.append(False)
mask = np.array(a)
print(e_on_direction_with_sheets(N, eps, mu, l, mask, L, 5.00, 0, 0))

In [ ]:
for N in [1, 2, 3]:
    f = 5
    eps = np.ones(2*N + 1)
    mu = np.ones(2*N +1)
    L = np.array([7.99446555e-9]*(2*N+1))
    l = np.array([0.00614498]*(2*N+1))
    l[0] = 0.02557249
    l[2*N] = 0
    a = ([False, True]*N)
    a.append(False)
    mask = np.array(a)

    e_norm = np.abs(e_on_direction_with_sheets(N, eps, mu, l, mask, L, 5, 0, 0))
    print('e_norm =', e_norm)
    #radiation_pattern_e_plane = lambda theta: np.abs(e_on_direction(N, eps, mu, l, theta = theta, phi=0)/max_e)**2
    radiation_pattern = lambda phi, theta: np.abs(e_on_direction_with_sheets(N, eps, mu, l, mask, L, theta = theta, phi=phi, f=f)/e_norm)**2 * np.sin(theta)
    res, err = integrate.dblquad(
        radiation_pattern,
        0, np.pi/2*0.99,                # границы для x
        lambda x: 0,          # нижняя граница y
        lambda x: np.pi*2*0.99,       # верхняя граница y
        epsrel=1e-2
    )
    #res_e, err_e = integrate.quad(radiation_pattern_e_plane, -np.pi/2, np.pi/2)
    #print('Directivity по E-плоскости =', 10*np.log10(2*np.pi/res_e), 'dBi')
    print(res, err)
    print('Directivity =', 10*np.log10(4*np.pi/res), 'dBi')
    theta = np.linspace(-np.pi/2*0.99, np.pi/2*0.99, 200)
    P = np.zeros(200)
    for i in range(len(theta)):
        P[i] = np.abs(e_on_direction_with_sheets(N, eps, mu, l, mask, L, theta = theta[i], phi=0, f=f))**2
    plt.plot(theta*180/np.pi, 10*np.log10(P/np.max(P)), label=f'N={N}')
plt.ylim(-60, 0)
plt.legend()
plt.show()

In [ ]:
directivities = []
f_arr = np.linspace(4.80, 5.05, 30)
for f in f_arr:
    N = 3
    eps = np.ones(2*N + 1)
    mu = np.ones(2*N +1)
    L = np.array([7.99446555e-9]*(2*N+1))
    l = np.array([0.00614498]*(2*N+1))
    l[0] = 0.02557249
    l[2*N] = 0
    a = ([False, True]*N)
    a.append(False)
    mask = np.array(a)

    e_norm = np.abs(e_on_direction_with_sheets(N, eps, mu, l, mask, L, f, 0, 0))
    #print('e_norm =', e_norm)
    #radiation_pattern_e_plane = lambda theta: np.abs(e_on_direction(N, eps, mu, l, theta = theta, phi=0)/max_e)**2
    radiation_pattern = lambda phi, theta: np.abs(e_on_direction_with_sheets(N, eps, mu, l, mask, L, theta = theta, phi=phi, f=f)/e_norm)**2 * np.sin(theta)
    res, err = integrate.dblquad(
        radiation_pattern,
        0, np.pi/2*0.99,                # границы для x
        lambda x: 0,          # нижняя граница y
        lambda x: np.pi*2*0.99,       # верхняя граница y
        epsrel=1e-2
    )
    #res_e, err_e = integrate.quad(radiation_pattern_e_plane, -np.pi/2, np.pi/2)
    #print('Directivity по E-плоскости =', 10*np.log10(2*np.pi/res_e), 'dBi')
    #print(res, err)
    dir = 10*np.log10(4*np.pi/res)
    directivities.append(dir)
    #print('Directivity =', dir, 'dBi', 'at f =', f, 'GHz')
plt.plot(f_arr, directivities)
plt.title('N=3')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Directivity (dBi)')
plt.show()

In [ ]:
directivities = []
f_arr = np.linspace(4.7, 5.1, 30)
for f in f_arr:
    N = 2
    eps = np.ones(2*N + 1)
    mu = np.ones(2*N +1)
    L = np.array([7.99446555e-9]*(2*N+1))
    l = np.array([0.00614498]*(2*N+1))
    l[0] = 0.02557249
    l[2*N] = 0
    a = ([False, True]*N)
    a.append(False)
    mask = np.array(a)

    e_norm = np.abs(e_on_direction_with_sheets(N, eps, mu, l, mask, L, f, 0, 0))
    #print('e_norm =', e_norm)
    #radiation_pattern_e_plane = lambda theta: np.abs(e_on_direction(N, eps, mu, l, theta = theta, phi=0)/max_e)**2
    radiation_pattern = lambda phi, theta: np.abs(e_on_direction_with_sheets(N, eps, mu, l, mask, L, theta = theta, phi=phi, f=f)/e_norm)**2 * np.sin(theta)
    res, err = integrate.dblquad(
        radiation_pattern,
        0, np.pi/2*0.99,                # границы для x
        lambda x: 0,          # нижняя граница y
        lambda x: np.pi*2*0.99,       # верхняя граница y
        epsrel=1e-2
    )
    #res_e, err_e = integrate.quad(radiation_pattern_e_plane, -np.pi/2, np.pi/2)
    #print('Directivity по E-плоскости =', 10*np.log10(2*np.pi/res_e), 'dBi')
    #print(res, err)
    dir = 10*np.log10(4*np.pi/res)
    directivities.append(dir)
    #print('Directivity =', dir, 'dBi', 'at f =', f, 'GHz')
plt.plot(f_arr, directivities)
plt.title('N=2')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Directivity (dBi)')
plt.show()

полоса примерно 6%

In [ ]:
directivities = []
f_arr = np.linspace(4.95, 5.05, 30)
for f in f_arr:
    N = 4
    eps = np.ones(2*N + 1)
    mu = np.ones(2*N +1)
    L = np.array([7.99446555e-9]*(2*N+1))
    l = np.array([0.00614498]*(2*N+1))
    l[0] = 0.02557249
    l[2*N] = 0
    a = ([False, True]*N)
    a.append(False)
    mask = np.array(a)

    e_norm = np.abs(e_on_direction_with_sheets(N, eps, mu, l, mask, L, f, 0, 0))
    #print('e_norm =', e_norm)
    #radiation_pattern_e_plane = lambda theta: np.abs(e_on_direction(N, eps, mu, l, theta = theta, phi=0)/max_e)**2
    radiation_pattern = lambda phi, theta: np.abs(e_on_direction_with_sheets(N, eps, mu, l, mask, L, theta = theta, phi=phi, f=f)/e_norm)**2 * np.sin(theta)
    res, err = integrate.dblquad(
        radiation_pattern,
        0, np.pi/2*0.99,                # границы для x
        lambda x: 0,          # нижняя граница y
        lambda x: np.pi*2*0.99,       # верхняя граница y
        epsrel=1e-2
    )
    #res_e, err_e = integrate.quad(radiation_pattern_e_plane, -np.pi/2, np.pi/2)
    #print('Directivity по E-плоскости =', 10*np.log10(2*np.pi/res_e), 'dBi')
    #print(res, err)
    dir = 10*np.log10(4*np.pi/res)
    directivities.append(dir)
    #print('Directivity =', dir, 'dBi', 'at f =', f, 'GHz')
plt.plot(f_arr, directivities)
plt.title('N=4')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Directivity (dBi)')
plt.show()

In [ ]:
for N in [2, 4, 6]:
    ###
    eps = np.ones(2*N + 1)
    mu = np.ones(2*N +1)
    L = np.array([7.99446555e-9]*(2*N+1))
    l = np.array([0.00614498]*(2*N+1))
    l[0] = 0.02557249
    l[2*N] = 0
    a = ([False, True]*N)
    a.append(False)
    mask = np.array(a)
    print(e_on_direction_array_with_sheets(N, eps, mu, l, mask, L, np.array([5, 5.1]), np.array([0, 0.1]), np.array([0, 0.1])))
    ###
    e_norm = np.abs(e_on_direction(N, eps, mu, l, theta = 0, phi=0, f=f))
    # theta_arr = np.linspace(-np.pi/2, np.pi/2, 500)
    # phi_arr = np.linspace(0, 2*np.pi, 500)
    # max_e = np.max(np.abs(e_on_direction_array(N, eps, mu, l, np.array([f]), phi_arr, theta_arr)[0]))
    #print('max_e =', max_e)
    print('e_norm =', e_norm)
    #radiation_pattern_e_plane = lambda theta: np.abs(e_on_direction(N, eps, mu, l, theta = theta, phi=0)/max_e)**2
    radiation_pattern = lambda phi, theta: np.abs(e_on_direction(N, eps, mu, l, theta = theta, phi=phi, f=f)/e_norm)**2 * np.sin(theta)
    res, err = integrate.dblquad(
        radiation_pattern,
        0, np.pi/2,                # границы для x
        lambda x: 0,          # нижняя граница y
        lambda x: np.pi*2,       # верхняя граница y
        epsrel=1e-2
    )
    #res_e, err_e = integrate.quad(radiation_pattern_e_plane, -np.pi/2, np.pi/2)
    #print('Directivity по E-плоскости =', 10*np.log10(2*np.pi/res_e), 'dBi')
    print('for N =', N, 'суммарная длина', sum(l)*1000, 'mm')
    print(res, err)
    print('Directivity =', 10*np.log10(4*np.pi/res), 'dBi')
    theta = np.linspace(-np.pi/2, np.pi/2, 200)
    P = np.zeros(200)
    for i in range(len(theta)):
        P[i] = np.abs(e_on_direction(N, eps, mu, l, theta = theta[i], phi=0, f=f))**2
    plt.plot(theta*180/np.pi, 10*np.log10(P/np.max(P)), label=f'N={N}')
plt.ylim(-60, 0)
plt.legend()
plt.show()

In [ ]:
def directivity_vs_freq(f_arr, N, mode='total'):
    directivities = np.zeros_like(f_arr)
    for i in range(len(f_arr)):
        f = f_arr[i]
        eps = np.ones(N)
        mu = np.ones(N)
        eps[1::2] = 10
        mu[::2] = 4
        l = l_for_article_resonance_structure(N, eps, mu, f=5, theta=0)
        e_norm = np.abs(e_on_direction(N, eps, mu, l, theta=0, phi=0, f=f))
        def e(v):
            phi, theta = v
            return -np.abs(e_on_direction(N, eps, mu, l, theta=theta, phi=phi, f=f))
        max_e = optimize.minimize(e, x0=(0.5, 0.5), bounds=((0, 2*np.pi), (-np.pi/2, np.pi/2))).fun * -1
        # theta_arr = np.linspace(-np.pi/2, np.pi/2, 500)
        # phi_arr = np.linspace(0, 2*np.pi, 500)
        # max_e = np.max(np.abs(e_on_direction_array(N, eps, mu, l, np.array([f]), phi_arr, theta_arr)[0]))
        if mode == 'total':
            radiation_pattern = lambda phi, theta: np.abs(e_on_direction(N, eps, mu, l, theta=theta, phi=phi, f=f)/max_e)**2 * np.sin(theta)
        if mode == 'normal':
            radiation_pattern = lambda phi, theta: np.abs(e_on_direction(N, eps, mu, l, theta=theta, phi=phi, f=f)/e_norm)**2 * np.sin(theta)
        res, err = integrate.dblquad(
            radiation_pattern,
            0, np.pi/2,                # границы для x
            lambda x: 0,          # нижняя граница y
            lambda x: np.pi*2,          # верхняя граница y
            epsrel=1e-2
        )
        directivities[i] = 10*np.log10(4*np.pi/res)
    return directivities

In [ ]:
f_arr = np.linspace(4.95, 5.05, 150)
for N in [2, 4, 6]:
    directivities = directivity_vs_freq(f_arr, N, mode='total')
    plt.plot(f_arr, directivities, label=f'N={N}')
    plt.xlabel('Frequency (GHz)')
    plt.ylabel('Directivity (dBi)')
    plt.legend()
    plt.show()

In [ ]:
f_arr = np.linspace(4.95, 5.05, 150)
for N in [4]:
    directivities = directivity_vs_freq(f_arr, N, mode='normal')
    plt.plot(f_arr, directivities, label=f'N={N}')
    plt.xlabel('Frequency (GHz)')
    plt.ylabel('Directivity (dBi)')
    plt.legend()
    plt.show()

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6, 7])
print(np.meshgrid(a, b))

In [ ]:
import plotly.graph_objects as go

theta = np.linspace(-np.pi/2, np.pi/2, 100)
phi = np.linspace(0, 2*np.pi, 100)
theta, phi = np.meshgrid(theta, phi)

R = np.abs(np.cos(theta))**2

X = R * np.sin(theta) * np.cos(phi)
Y = R * np.sin(theta) * np.sin(phi)
Z = R * np.cos(theta)

fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, colorscale="Viridis")])
fig.show()   # откроется интерактивный график


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
def truncate_colormap(cmap, minval=0.1, maxval=0.9, n=256):
    """Обрезает colormap до нужного диапазона"""
    new_cmap = LinearSegmentedColormap.from_list(
        f'trunc({cmap.name},{minval:.2f},{maxval:.2f})',
        cmap(np.linspace(minval, maxval, n))
    )
    return new_cmap


def radiation_pattern_plot_3d(f, phi_in, theta_in, logscale=True, output='notebook', title=''):

    # Берём jet и обрезаем края
    cmap = truncate_colormap(plt.get_cmap("hsv_r"), 0.35, 0.97)
    colorscale = []
    for i, c in enumerate(cmap(np.linspace(0, 1, 256))):
        colorscale.append([i/255, f'rgb({int(c[0]*255)},{int(c[1]*255)},{int(c[2]*255)})'])
    # открывать график в ноутбуке
    pio.renderers.default = output
    theta = theta_in
    phi = phi_in
    R = f
    # Нормировка шкалы начало
    max_e = np.max(np.abs(f))
    radiation_pattern = lambda phi, theta: np.abs(e_on_direction(N, eps, mu, l, theta = theta, phi=phi))**2 * np.sin(theta) #это нужно оптимизировать будет
    total_energy, err = integrate.dblquad(
        radiation_pattern,
        0, np.pi/2,                # границы для x
        lambda x: 0,          # нижняя граница y
        lambda x: np.pi*2          # верхняя граница y
    )
    print('Directivity = ',10*np.log10(4*np.pi*max_e**2/total_energy))
    R = abs(R)
    threshhold = np.sqrt(total_energy/(4*np.pi))
    mask = R>threshhold
    mask_2 = R<=threshhold
    R[mask_2] = 0
    R[mask] = 20*np.log10(R[mask]/threshhold)
    # if logscale:
    #     R = log_scale_dB(R)
    colorbar=dict(
        title=dict(
            text="[dBi]",      # заголовок
            side="top"              # сторона (left / right / top / bottom)
        )
        # tickvals=[60, 20, 0],       # где ставить подписи
        # ticktext=["-40 дБ", "-20 дБ", "0 дБ"]  # сами подписи
    )
    # Нормировка шкалы конец
    # Название графика
    title=dict(
        text=title,
        x=0.5,   # выравнивание по центру (0 = слева, 0.5 = центр, 1 = справа)
        xanchor="center",
        font=dict(size=20)  # размер шрифта
    )
    theta, phi = np.meshgrid(theta, phi)
    X = R * np.sin(theta) * np.cos(phi)
    Y = R * np.sin(theta) * np.sin(phi)
    Z = R * np.cos(theta)
    fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, surfacecolor=R, colorscale=colorscale, colorbar = colorbar, lighting=dict(ambient=1, diffuse=0, specular=0, roughness=1))])
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(range=[X.min(), X.max()], title="X"),
            yaxis=dict(range=[Y.min(), Y.max()], title="Y"),
            zaxis=dict(range=[Z.min(), Z.max()], title="Z"),
            aspectmode="data"   # оси масштабируются по данным
        )
    )
    fig.show()

In [ ]:
for N in [2, 4, 6]:
    eps = np.ones(N)
    mu = np.ones(N)
    eps[1::2] = 10
    mu[::2] = 4
    l = l_for_article_resonance_structure(N, eps, mu, f=5, theta = 0)
    print(l*1000)
    phi = np.linspace(0, 2*np.pi, 100)
    theta = np.concatenate((np.linspace(0, np.pi/30, 100), np.linspace(np.pi/30, np.pi/2, 100)))
    f = e_on_direction_array(N, eps, mu, l, np.array([5]), phi, theta)[0]
    radiation_pattern_plot_3d(f, phi, theta, title=f'N={N}, total higth={sum(l)*1000:.1f} mm', output='browser')

In [ ]:
phi_test = np.linspace(-np.pi/4, 3*np.pi/4, 100)[:, None]
theta_test = np.linspace(0, np.pi, 100)[None, :]
phi_ones = np.ones_like(phi_test)
eee = np.abs(np.sin(theta_test))*phi_ones

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"

# 1. Создаём обрезанный colormap в matplotlib
from matplotlib.colors import LinearSegmentedColormap

def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=256):
    new_cmap = LinearSegmentedColormap.from_list(
        f'trunc({cmap.name},{minval:.2f},{maxval:.2f})',
        cmap(np.linspace(minval, maxval, n))
    )
    return new_cmap

cmap = truncate_colormap(plt.get_cmap("hsv_r"), 0.3, 1)

# 2. Преобразуем colormap в формат Plotly
colorscale = []
for i, c in enumerate(cmap(np.linspace(0, 1, 256))):
    colorscale.append([i/255, f'rgb({int(c[0]*255)},{int(c[1]*255)},{int(c[2]*255)})'])

# 3. Сетка для ДН
theta = np.linspace(0, np.pi, 100)
phi = np.linspace(-np.pi/4, 3*np.pi/4, 100)
theta, phi = np.meshgrid(theta, phi)

R = log_scale_dB(eee)
#R = log_scale_dB(R)
X = R * np.sin(theta) * np.cos(phi)
Y = R * np.sin(theta) * np.sin(phi)
Z = R * np.cos(theta)

# 4. Построение поверхности
fig = go.Figure(data=[go.Surface(
    x=X, y=Y, z=Z,
    surfacecolor=R,
    colorscale=colorscale,
    lighting=dict(ambient=1, diffuse=0, specular=0, roughness=1)
)])

fig.show()